# Notebook 01 · La idea adversaria: una GAN que aprende una distribución

**Introducción al Deep Learning — Módulo 3, Unidad 3: GANs (Parte I)**

Entendamos la **mecánica adversaria** con el ejemplo más simple posible: una GAN que aprende a generar
números de una **distribución objetivo** (una campana de Gauss).

- **Generador**: convierte ruido aleatorio en números.
- **Discriminador**: decide si un número es real o falso.
- Ambos compiten y mejoran a la vez.

> 💡 En Colab, TensorFlow ya viene instalado. Ejecuta las celdas en orden con **Shift + Enter**.

## 1. La distribución real que queremos imitar

In [ ]:
try:
    import tensorflow as tf
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow"])
    import tensorflow as tf
import numpy as np, matplotlib.pyplot as plt
tf.random.set_seed(42); np.random.seed(42)

MEDIA, DESV = 4.0, 0.5
def datos_reales(n):
    return np.random.normal(MEDIA, DESV, size=(n, 1)).astype("float32")

plt.figure(figsize=(7, 3))
plt.hist(datos_reales(2000), bins=40, color="#4355FF", alpha=0.7)
plt.title("Distribución real objetivo  N(4, 0.5)"); plt.tight_layout(); plt.show()

## 2. Generador y discriminador

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

DIM_RUIDO = 8
generador = Sequential([Input(shape=(DIM_RUIDO,)),
                        Dense(16, activation="relu"), Dense(16, activation="relu"), Dense(1)])
discriminador = Sequential([Input(shape=(1,)),
                            Dense(16, activation="relu"), Dense(1, activation="sigmoid")])
bce = tf.keras.losses.BinaryCrossentropy()
opt_g = tf.keras.optimizers.Adam(0.001); opt_d = tf.keras.optimizers.Adam(0.001)
print("Modelos creados.")

## 3. El bucle de entrenamiento adversario

En cada paso entrenamos el **discriminador** (real=1, falso=0) y luego el **generador** (que quiere que sus
falsos se clasifiquen como reales).

In [ ]:
def ruido(n):
    return tf.random.normal((n, DIM_RUIDO))

@tf.function
def paso(reales):
    n = tf.shape(reales)[0]
    falsos = generador(ruido(n))
    with tf.GradientTape() as td:
        loss_d = bce(tf.ones_like(discriminador(reales)), discriminador(reales)) \
               + bce(tf.zeros_like(discriminador(falsos)), discriminador(falsos))
    opt_d.apply_gradients(zip(td.gradient(loss_d, discriminador.trainable_variables),
                             discriminador.trainable_variables))
    with tf.GradientTape() as tg:
        f = generador(ruido(n))
        loss_g = bce(tf.ones_like(discriminador(f)), discriminador(f))
    opt_g.apply_gradients(zip(tg.gradient(loss_g, generador.trainable_variables),
                             generador.trainable_variables))
    return loss_d, loss_g

for i in range(1200):
    ld, lg = paso(datos_reales(128))
    if i % 300 == 0:
        print(f"paso {i:4d} | loss_D {ld.numpy():.3f} | loss_G {lg.numpy():.3f}")
print("Entrenamiento terminado.")

## 4. ¿Ha aprendido el generador la distribución?

In [ ]:
generados = generador(ruido(2000)).numpy()
plt.figure(figsize=(7, 4))
plt.hist(datos_reales(2000), bins=40, color="#4355FF", alpha=0.6, label="reales")
plt.hist(generados, bins=40, color="#FF647E", alpha=0.6, label="generados (GAN)")
plt.title("Reales vs. generados"); plt.legend(); plt.tight_layout(); plt.show()
print("Media real ~ %.2f | Media generada ~ %.2f" % (MEDIA, generados.mean()))

El histograma generado se aproxima al real: el generador aprende a imitar la distribución **sin verla
directamente**, solo a través del veredicto del discriminador.

## Resumen

- Una GAN entrena **dos redes que compiten**: generador (crea) y discriminador (juzga).
- El generador parte de **ruido** y aprende de la **retroalimentación** del discriminador.
- Tras el entrenamiento, los datos generados se parecen a los reales.

➡️ En el **Notebook 02** aplicaremos la misma idea a una **nube de puntos en 2D**.